# 0. Define Taxonomies

Defines every classification taxonomy this project knows about and lands
them in Unity Catalog so the rest of the pipeline (and the app) can read
them uniformly.

Outputs:
- `<name>_taxonomy` (one per schema, normalized to `code, label, level_path` + raw cols)
- `schema_registry` — one row per schema with metadata
- `cat_predictions` — empty unified predictions table

Add a new taxonomy by writing a `ClassificationSchema` subclass in
`src/schemas/` and registering it. This notebook needs no code change.

In [ ]:
%pip install pyyaml openai openpyxl pydantic
%restart_python

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import pyspark.sql.functions as F
from src.utils import get_spark
from src.config import load_config
from src.schemas import STRATEGY, list_schemas

spark = get_spark()
config = load_config()

In [ ]:
dbutils.widgets.removeAll()
dbutils.widgets.text('catalog', config.catalog)
dbutils.widgets.text('schema', config.schema_name)
catalog = dbutils.widgets.get('catalog')
schema = dbutils.widgets.get('schema')
spark.sql(f'USE {catalog}.{schema}')

## Load each taxonomy

In [ ]:
import re

def _sanitize(col: str) -> str:
    return re.sub(r'[^A-Za-z0-9_]+', '_', col).strip('_').lower()

registry_rows = []
for s in list_schemas():
    print(f'==> {s.name}: {s.display_name}')
    pdf = s.load_taxonomy()
    pdf['level_path'] = pdf['level_path'].apply(lambda xs: [str(x) for x in xs])
    for c in [c for c in pdf.columns if c not in ('code','label','level_path')]:
        pdf[c] = pdf[c].astype(str).fillna('')
    # Delta rejects spaces / special chars in column names.
    pdf = pdf.rename(columns={c: _sanitize(c) for c in pdf.columns})

    target = f'{catalog}.{schema}.{s.name}_taxonomy'
    (spark.createDataFrame(pdf)
        .withColumn('schema_name', F.lit(s.name))
        .write.format('delta').mode('overwrite').option('overwriteSchema','true')
        .saveAsTable(target))
    print(f'   {len(pdf):,} rows -> {target}')
    registry_rows.append({
        'name': s.name,
        'display_name': s.display_name,
        'description': s.spec.description,
        'classify_method': STRATEGY.get(s.name, 'ai_classify'),
        'taxonomy_table': f'{s.name}_taxonomy',
        'level_columns': list(s.spec.level_columns),
        'code_column': s.spec.code_column,
        'label_column': s.spec.label_column,
        'description_columns': list(s.spec.description_columns),
        'leaf_count': int(len(pdf)),
    })

## Publish the schema registry

In [ ]:
registry_pdf = pd.DataFrame(registry_rows)
(spark.createDataFrame(registry_pdf)
    .write.format('delta').mode('overwrite').option('overwriteSchema','true')
    .saveAsTable(f'{catalog}.{schema}.schema_registry'))
registry_pdf

## Create the unified `cat_predictions` table

Every classifier (ai_classify, vector search, agent review, etc.)
MERGEs into this table keyed on `(order_id, schema_name, source)`.

In [ ]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{schema}.cat_predictions (
    order_id STRING,
    schema_name STRING,
    code STRING,
    label STRING,
    level_path ARRAY<STRING>,
    confidence DOUBLE,
    rationale STRING,
    source STRING,
    candidates ARRAY<STRUCT<code: STRING, label: STRING, score: DOUBLE>>,
    classified_at TIMESTAMP
) USING DELTA
""")
spark.sql(f'DESCRIBE {catalog}.{schema}.cat_predictions').display()